# 4-Channel YOLOv8 (Early Fusion) for RGB-T Drone Imagery
This notebook trains a 4-channel early-fusion YOLOv8 model.

In [ ]:
!pip install ultralytics -q
import ultralytics

In [ ]:
import cv2
import torch
import numpy as np
from ultralytics import YOLO
import ultralytics.data.base as base

# 1. Patch Ultralytics imread to keep the 4th (thermal) channel
base.imread = lambda p, flags=cv2.IMREAD_COLOR: cv2.imdecode(np.fromfile(p, np.uint8), cv2.IMREAD_UNCHANGED)

# 2. Create a custom YOLOv8 config that expects 4 input channels
custom_yaml = 'yolov8n-4ch.yaml'
with open(custom_yaml, 'w') as f:
    f.write('''
nc: 4 # person, crowd, rider, uncertain
channels: 4 # RGB + Thermal
scales:
  n: [0.33, 0.25, 1024]
backbone:
  - [-1, 1, Conv, [64, 3, 2]]  # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]]  # 1-P2/4
  - [-1, 3, C2f, [128, True]]
  - [-1, 1, Conv, [256, 3, 2]]  # 3-P3/8
  - [-1, 6, C2f, [256, True]]
  - [-1, 1, Conv, [512, 3, 2]]  # 5-P4/16
  - [-1, 6, C2f, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]  # 7-P5/32
  - [-1, 3, C2f, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]  # 9
head:
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 6], 1, Concat, [1]]  # cat backbone P4
  - [-1, 3, C2f, [512]]  # 12
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 4], 1, Concat, [1]]  # cat backbone P3
  - [-1, 3, C2f, [256]]  # 15 (P3/8-small)
  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 12], 1, Concat, [1]]  # cat head P4
  - [-1, 3, C2f, [512]]  # 18 (P4/16-medium)
  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 9], 1, Concat, [1]]  # cat head P5
  - [-1, 3, C2f, [1024]]  # 21 (P5/32-large)
  - [[15, 18, 21], 1, Detect, [nc]]  # Detect(P3, P4, P5)
    ''')

# 3. Build model from YAML and safely load matching layers from pretrained weights
model = YOLO(custom_yaml).load('yolov8n.pt')

# 4. First conv: copy the pretrained RGB filters, init the thermal channel from their mean
w3 = YOLO('yolov8n.pt').model.model[0].conv.weight.data   # [16, 3, 3, 3]
with torch.no_grad():
    c = model.model.model[0].conv.weight                   # [16, 4, 3, 3]
    c[:, :3] = w3
    c[:, 3:] = w3.mean(1, keepdim=True)
print('Successfully initialized 4-channel first layer using mean of RGB filters!')

# 5. Start Training!
# - Disabled HSV augmentations as they corrupt 4-channel data
# - Increased imgsz to 1024 to catch small drone objects
results = model.train(
    data='/kaggle/input/yolo-rgbt-4ch/dataset.yaml',
    epochs=25,
    imgsz=1024,
    batch=16,
    device=0,
    hsv_h=0, hsv_s=0, hsv_v=0
)